In [2]:
import pandas as pd
import glob
from time import sleep
import urllib.parse
from requests import get
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from datetime import datetime
from bs4 import BeautifulSoup
import re
import random
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys


In [ ]:
# # NOT FUNCTIONAL automate getting lists from IBKR
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import NoSuchElementException, TimeoutException

# options = webdriver.ChromeOptions()
# prefs = {"profile.managed_default_content_settings.images": 2}
# options.add_experimental_option("prefs", prefs)

# driver = webdriver.Chrome(options=options)
# driver.maximize_window()

# url = f'https://www.interactivebrokers.com/en/trading/products-exchanges.php#/'
# driver.get(url)
# try:
#     button = driver.find_element(By.ID, 'gdpr-reject-all')
#     WebDriverWait(driver, 5).until(EC.element_to_be_clickable(button))
#     button.click()
# except NoSuchElementException:
#     pass

# time.sleep(3)
# WebDriverWait(driver, 10).until(EC.element_to_be_clickable(By.CSS_SELECTOR, '._btn.text-bold.after-4.text-left'))
# button = driver.find_element(By.CSS_SELECTOR, '._btn.text-bold.after-4.text-left')
# button.click()

# time.sleep(1)
# button = driver.find_element(By.CSS_SELECTOR, '.fa.fa-chevron-left')
# WebDriverWait(driver, 10).until(EC.element_to_be_clickable(button))
# button.click()

# time.sleep(1)
# button = driver.find_element(By.CSS_SELECTOR, '._btn.inversed-black.remove')
# driver.execute_script("window.scrollBy(0,400);")
# time.sleep(2)
# button.click()

# WebDriverWait(driver, 10).until(EC.presence_of_element_located(By.ID, '_dd158'))
# time.sleep(1)
# driver.execute_script("window.scrollBy(0,600);")
# button = driver.find_element(By.ID, '_dd158')
# button.click()

# # while True:
# #     try:
# #         WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".paginacion")))
# #         WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "anuncio")))
# #         html = driver.page_source
# #         soup = BeautifulSoup(html, 'html.parser')
# #         cars += soup.find_all('article', class_='anuncio')
        
# #         pages = driver.find_element(By.CLASS_NAME, "paginacion")
# #         next_button = pages.find_elements(By.TAG_NAME, "li")[-1]
# #         next_url = next_button.find_element(By.TAG_NAME, "a").get_attribute("href")
# #         driver.get(next_url)

# #     except NoSuchElementException:
# #         print('no such element')
# #         print(next_url)
# #         break
# #     except TimeoutException:
# #         print('timeout')
# #         print(next_url)
# #         break

# # driver.quit()

In [13]:
# Load ETF csvs
path = "data/etfs-IBKR/*.csv" 
all_files = glob.glob(path)

df_list = []
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0, encoding='utf-8')
    df_list.append(df)

df = pd.concat(df_list, axis=0, ignore_index=True)

df.columns = df.columns.str.lower()
df.description = df.description.replace('&amp;', '&', regex=True)

# Define your mapping dictionary
country_dict = {
    'TW': 'Taiwan',
    'US': 'United States',
    'DE': 'Germany',
    'LU': 'Luxembourg',
    'JP': 'Japan',
    'SG': 'Singapore',
    'JE': 'Jersey',
    'FR': 'France',
    'IE': 'Ireland',
    'AU': 'Australia',
    'CH': 'Switzerland',
    'HK': 'Hong Kong',
    'CA': 'Canada',
    'MX': 'Mexico',
    'IN': 'India',
    'IL': 'Israel',
    'LI': 'Liechtenstein',
    'NL': 'Netherlands',
    'NO': 'Norway',
    'KY': 'Cayman Islands',
    'SE': 'Sweden',
    'ES': 'Spain',
    'GG': 'Guernsey',
    'PL': 'Poland',
    'HU': 'Hungary',
    'XX': 'Other',
    'ZA': 'South Africa',
    'CN': 'China'
}
df['region'] = df['region'].map(country_dict)

In [14]:
# Create eur dataframe
eur = df.loc[df.currency == 'EUR']
eur = eur.drop('symbol', axis=1).copy()

eur['query'] = ( 'justETF etf-profile '
                + eur['description'] + ' ' 
                + eur['currency'] + ' ' 
                + eur['region'] + ' ' 
                + eur['exchange'])

In [104]:
def string_to_url(query):
    query = urllib.parse.quote_plus(query)  # URL encoding
    url = f"https://www.google.com/search?q={query}&hl=en"
    return url

remaining_queries = eur['query'].to_list()
saved_queries = []

In [186]:
# # Get Google links
# try:
#     for result in search_results:
#         try:
#             if result[0]['query'] in remaining_queries:
#                 remaining_queries.remove(result[0]['query'])
#         except (TypeError, IndexError):
#             continue
# except NameError:
#     pass

# try:
#     for result in saved_queries:
#         try:
#             if result[0]['query'] in remaining_queries:
#                 remaining_queries.remove(result[0]['query'])
#         except (TypeError, IndexError):
#             continue
# except NameError:
#     pass

# # saved_queries += search_results
# search_results = []
# print(len(saved_queries), len(remaining_queries), len(eur['query'].to_list()))

# driver = webdriver.Chrome()

# # Clear bot check
# def bot_check():
#     while True:
#         sleep(2)
#         user_input = input()
#         if user_input:
#             break

# driver.get('https://www.google.com/search?q=test')
# bot_check()
# # Clear cookies
# WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".QS5gu.sy4vM")))
# sleep(random.uniform(0, .1))
# button = driver.find_element(By.CSS_SELECTOR, ".QS5gu.sy4vM")
# driver.execute_script("arguments[0].scrollIntoView();", button)
# sleep(random.uniform(0, .5))
# button.click()
# i = 0
# for query in remaining_queries:
#     i += 1
#     result_data = []

#     driver.get(string_to_url(query))
#     try:
#         # WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".tfyYAe.HqgSZe")))
#         WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".Ap1Qsc")))
#     except TimeoutException:
#         bot_check()
#         print(i)
#         i = 0
#         # continue
#     sleep(random.uniform(0, .5))
#     for j in range(random.randint(0, 2)):
#         actions = ActionChains(driver)
#         sleep(random.uniform(0, .5))
#         actions.send_keys(Keys.PAGE_DOWN).perform()

#     html = driver.page_source
#     soup = BeautifulSoup(html, 'html.parser')
#     results = soup.find_all('div', class_='yuRUbf')

#     for result in results:
#         link = result.find('a')['href']
#         if 'etf-profile' in link:
#             etf_profile = {
#                 'link': link,
#                 'heading': result.find('h3').text,
#                 'query': query
#             }
#             result_data.append(etf_profile)
#         else:
#             etf_profile = {
#                 'link': None,
#                 'heading': None, 
#                 'query': query
#             }
#             result_data.append(etf_profile)

#     search_results.append(result_data)
#     sleep(random.uniform(1, 2))

In [164]:
# # Cold-store saved_queries
# saved2 = []
# for search in saved_queries:
#     saved3 = []
#     for result in search:
#         if result['link'] == None:
#             pass
#         else:
#             saved3.append(result)
#     saved2.append(saved3)

# Save progress as json
# import json

# # Open the file in write mode ('w')
# with open("saved2.json", "w") as file:
#     # Use json.dump to write the list to a file
#     json.dump(saved2, file)

# print("List has been written to 'my_list.json'")

# flat_list = [item for sublist in saved2 for item in sublist]

# # Convert the list into a DataFrame
# df_list = pd.DataFrame(flat_list)
# df_list.to_csv('etf_urls.csv', index=False)

In [322]:
# reduce rearch results to unique urls to scrape
df = pd.read_csv('etf_urls.csv')
df = df.loc[~df['heading'].isin(['ETF Screener', 'Buscador de ETF', 'ETF Chart Comparison'])]
df = df.loc[df['link'].str.contains('justetf')]
df['heading'] = df['heading'].str.replace('(', '').str.replace(')', '')

df['rank'] = df.groupby('query').cumcount() + 1
df = df[df['rank'] <= 3]


# revert country_dict
inverted_country_dict = {v: k for k, v in country_dict.items()}

def replace_second_last_word(query, inverted_country_dict):
    words = query.split()
    if words[-2] in inverted_country_dict:
        words[-2] = inverted_country_dict[words[-2]]
    return ' '.join(words)

df['query'] = df['query'].apply(lambda x: replace_second_last_word(x, inverted_country_dict))

# adjust url
def change_url2en(link):
    words = link.split(r'/')
    if words[3] != 'en':
        words[3] = 'en'
    return '/'.join(words)

df['link'] = df['link'].apply(lambda x: change_url2en(x))

# replace other words
currency_dict = {'euro': 'EUR',
                 '...': '',
                 '|': '',
                 'government': 'GOV'
                 }

def replace_any_word(query, currency_dict):
    words = query.split()
    words = [currency_dict[word.lower()] if word.lower() in currency_dict else word for word in words]
    return ' '.join(words)

df['heading'] = df['heading'].apply(lambda x: replace_any_word(x, currency_dict))

# calculate jaccard
def jaccard_similarity(list1, list2):
    s1 = set(list1)
    s2 = set(list2)
    return len(s1.intersection(s2))

df['jaccard'] = df.apply(lambda row: jaccard_similarity(row['query'].lower().split(), row['heading'].lower().split()), axis=1)


df['score'] = df['jaccard'] - df['rank']
df = df.sort_values(by='score',ascending=False).drop_duplicates(subset='link')
df.shape

C:\Users\Alex\AppData\Local\Temp\ipykernel_16996\3650439290.py:5: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df['heading'] = df['heading'].str.replace('(', '').str.replace(')', '')


(2605, 6)

In [509]:
# scraping functions
def get_basics(soup):
    try:
        table = soup.find_all('table', class_='table etf-data-table')

        basics = {}
        performance = {}
        risk = {}
        for i, table in enumerate(table):
            data = {}
            for row in table.find_all('tr'):
                label_elem = row.find('td', class_='vallabel')
                if label_elem:
                    label = label_elem.text.strip()
                    label = [text.lower().strip() for text in label.split(' ')]
                    label = '_'.join(label)
                    value_elem = label_elem.find_next_sibling()
                    if value_elem:
                        value = value_elem.get_text(strip=True)
                        data[label] = value
            if i == 0:
                basics = data
            elif i == 1:
                performance = data
            elif i == 2:
                risk = data
            else:
                print('Unexpected table - extra data?')
        
        return basics, performance, risk
    
    except (AttributeError, ValueError, IndexError) as e:
        basics = {'index': None,
                'investment_focus': None,
                'fund_size': None,
                'total_expense_ratio': None,
                'replication': None,
                'legal_structure': None,
                'strategy_risk': None,
                'sustainability': None,
                'fund_currency': None,
                'currency_risk': None,
                'volatility_1_year_(in_eur)': None,
                'inception/_listing_date': None,
                'distribution_policy': None,
                'distribution_frequency': None,
                'fund_domicile': None,
                'fund_provider': None}
        performance = {'ytd': None,
                '1_month': None,
                '3_months': None,
                '6_months': None,
                '1_year': None,
                '3_years': None,
                '5_years': None,
                'since_inception_(max)': None,
                '2023': None,
                '2022': None,
                '2021': None,
                '2020': None}
        risk = {'volatility_1_year': None,
                'volatility_3_years': None,
                'volatility_5_years': None,
                'return_per_risk_1_year': None,
                'return_per_risk_3_years': None,
                'return_per_risk_5_years': None,
                'maximum_drawdown_1_year': None,
                'maximum_drawdown_3_years': None,
                'maximum_drawdown_5_years': None,
                'maximum_drawdown_since_inception': None,}
        print(f'    BASICS = {e}')
        
        return basics, performance, risk


def get_intro_data(soup):
    try:
        intro_elems = soup.find_all('div', class_='d-flex d-flex-column')

        intro = {}
        for elem in intro_elems:
            label = elem.find('div', class_='vallabel').text.strip().lower()
            value = elem.find('div', class_='val bold').text.strip()
            intro[label.replace(' ','_')] = value

        return intro
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'INTRO = {e}')

        return {'TER': None,
                'Distribution policy': None,
                'Replication': None,
                'Fund size': None,
                'Holdings': None,}


def get_top_10(soup):
    try:
        # Get top 10 elem + num holdings
        top_10_elem = soup.find('div', class_='constituents-top mb-3 clearfix') # weight top 10
        num_holdings = top_10_elem.find('div', class_='left').find_all('div')[1]
        num_holdings = num_holdings.get_text().split(' ')[-1]
        num_holdings = int(num_holdings.replace(',','').replace('.',''))
        
        top_10_percentage = top_10_elem.find('div', class_='right').find('span')
        top_10_percentage = top_10_percentage.get_text().strip('%')
        top_10_percentage = float(top_10_percentage.replace(',','')) / 100

        return {'top_10_holdings': top_10_percentage, 'num_holdings': num_holdings}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    TOP 10 HOLDINGS = {e} 10')
        
        return {'top_10_holdings': None, 'num_holdings': None}


def get_current_dividend(soup):
    try:
        table = soup.find('table', class_='table m-0')
        
        current_dividend = {}
        for row in table.find_all('tr'):
            label_elem = row.find('td', class_='vallabel')
            if label_elem:
                label = label_elem.text.strip()
                value_elem = label_elem.find_next_sibling()
                if value_elem:
                    value = value_elem.get_text(strip=True)
                    current_dividend[label.lower().replace(' ','_')] = value

        return current_dividend
    
    except (AttributeError, ValueError, IndexError) as e:
        # print(f'    CURRENT_DIVIDENDS = {e}')
        
        return {'current_dividend_yield': None, 'dividends_(last_12_months)': None},


def get_historic_dividend(soup):
    try:
        table = soup.find('table', class_='table m-0 mobile-table')
        
        historic_dividend = []
        for row in table.find_all('tr'):
            cols = row.find_all('td')
            if cols:
                period = cols[0].text.strip()
                dividend = cols[1].text.strip()
                yield_percent = cols[2].text.strip()
                historic_dividend.append((period, dividend, yield_percent))

        return {'historic_dividend': historic_dividend}

    except (AttributeError, ValueError, IndexError) as e:
        # print(f'    HISTORIC_DIVIDENDS = {e}')

        return {'historic_dividend':[]}


def get_exchanges(soup):
    try:
        table = soup.find('table', class_='table mobile-table')

        exchanges = []
        for row in table.find_all('tr'):
            cols = row.find_all('td')
            if cols:
                period = cols[0].text.strip()
                dividend = cols[1].text.strip()
                yield_percent = cols[2].text.strip()
                bloomberg_inav = cols[3].text.strip()
                reuters_inav = cols[4].text.strip()
                market_maker = cols[5].text.strip()
                exchanges.append((period, dividend, yield_percent, bloomberg_inav, reuters_inav, market_maker))

        return {'exchanges': exchanges}
        
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    EXCHANGES = {e}')

        return {'exchanges': []}



def get_error(soup):
    try:
        error_elem = soup.find('span', class_='feedbackPanelERROR')
        if 'fund has been liquidated or merged' in error_elem.text:
            return {'liquidated': True}
        else:
            return {'liquidated': False}
        
    except AttributeError as e:
        # print(f'    ERRORS = {e}')

        return {'liquidated': False}


def get_pdfs(soup):
    try:
        heading_elems = soup.find_all('h3')

        for heading in heading_elems:
            if heading.text.strip() == 'Documents':
                table = heading.parent.find('table', class_='table mb-0')
                pdf_elem = table.find_all('a', class_='download-link')

                pdfs = {}
                for pdf in pdf_elem:
                    pdf_text = re.sub(r'\s+\(', '_', pdf.text.strip())
                    pdf_text = re.sub(r'\)', '', pdf_text).lower()
                    pdfs[pdf_text] = pdf['href']
                    
                return pdfs
            
        return {'factsheet_en': None,
                'kid_de': None,
                'prospectus_en': None,
                'semi-annual report_en': None,
                'annual report_en': None}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    PDFS = {e} 10')
    
        return {'factsheet_en': None,
                'kid_de': None,
                'prospectus_en': None,
                'semi-annual report_en': None,
                'annual report_en': None}
    

def get_indexNcategories(soup):
    try:
        heading_elems = soup.find_all('div', class_='h4')

        for heading in heading_elems:
            if heading.text.strip() == 'Similar ETFs via ETF search':
                table = heading.parent.find('table', class_='table mb-0')

                indexNcategories = {}
                for row in table.find_all('tr'):
                    text = row.text.strip().split(':')
                    indexNcategories[text[0].lower()] = text[1].strip()
                    
                return indexNcategories
        return {'index': None, 'category': None}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    INDEX+CATEGORIES = {e}')
        return {'index': None, 'categories': None}


def get_holdings(soup):
    try:
        heading_elems = soup.find_all('h3')
        
        for heading in heading_elems:
            if heading.text.strip() == 'Top 10 Holdings':
                table = heading.parent.find('table', class_='table mb-0')

                holdings = []
                for row in table.find_all('tr'):
                    code_elem = row.find('span')
                    percentage_elem = row.find('div', class_='right ws').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        holdings.append((code, percentage))

                return {'holdings': holdings}
        return {'holdings': []}
        
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    HOLDINGS = {e}')

        return {'holdings': []}


def get_countries(soup):
    try:
        heading_elems = soup.find_all('h3')
        
        for heading in heading_elems:
            if heading.text.strip() == 'Countries':
                table = soup.find_all('table', class_='table mb-0')[-2]

                countries = []
                for row in table.find_all('tr'):
                    code_elem = row.find('td')
                    percentage_elem = row.find('div', class_='right').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        countries.append((code, percentage))

                return {'countries': countries}
        return {'countries': []}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    COUNTRIES = {e}')
        
        return{'countries': []}
        

def get_sectors(soup):
    try:
        heading_elems = soup.find_all('h3')

        for heading in heading_elems:
            if heading.text.strip() == 'Sectors':
                table = soup.find_all('table', class_='table mb-0')[-2]

                sectors = []
                for row in table.find_all('tr'):
                    code_elem = row.find('td')
                    percentage_elem = row.find('div', class_='right').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        sectors.append((code, percentage))

                return {'sectors': sectors}
        return {'sectors': []}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    SECTORS = {e}')

        return {'sectors': []}

In [511]:
# scrape justETF data
# url_data = []
# check existing data to aviod redundant requests
if url_data:
    url_set = set([sublist[0] for sublist in url_data])
    remaining_urls = df['link'].to_list()
    remaining_urls = [x for x in remaining_urls if x not in url_set]
else:
    url_data = []
    remaining_urls = df['link'].to_list()

counter = len(url_data)
for url in remaining_urls:
    print(f'{counter} == {url}')
    counter += 1

    response = get(url)
    if response.status_code != 200:
        print('got bad response !=200')
        break
    soup = BeautifulSoup(response.text, 'html.parser')

    basics, performance, risk = get_basics(soup)
    pdfs = get_pdfs(soup)
    indexNcategories = get_indexNcategories(soup)
    top10 = get_top_10(soup)
    holdings = get_holdings(soup)
    countries = get_countries(soup)
    sectors = get_sectors(soup)
    current_dividend = get_current_dividend(soup)
    historic_dividends = get_historic_dividend(soup)
    exchanges = get_exchanges(soup)
    intro_data = get_intro_data(soup)        
    error = get_error(soup)

    url_data.append([url, exchanges, risk, historic_dividends, current_dividend, performance, holdings, sectors, countries, top10, basics, indexNcategories, pdfs, error])

    sleep(1)

0 == https://www.justetf.com/en/etf-profile.html?isin=LU0671493277
1 == https://www.justetf.com/en/etf-profile.html?isin=IE0004J37T45
2 == https://www.justetf.com/en/etf-profile.html?isin=FR0010756114
    TOP 10 HOLDINGS = 'NoneType' object has no attribute 'find' 10
3 == https://www.justetf.com/en/etf-profile.html?isin=LU0879397742
4 == https://www.justetf.com/en/etf-profile.html?isin=LU0879399441
5 == https://www.justetf.com/en/etf-profile.html?isin=IE00BD34DJ91
6 == https://www.justetf.com/en/etf-profile.html?isin=LU0446734369
7 == https://www.justetf.com/en/etf-profile.html?isin=IE00BHXMHK04
8 == https://www.justetf.com/en/etf-profile.html?isin=IE00B7WK2W23
    TOP 10 HOLDINGS = 'NoneType' object has no attribute 'find' 10
9 == https://www.justetf.com/en/etf-profile.html?isin=IE00BD4TXV59
10 == https://www.justetf.com/en/etf-profile.html?isin=IE00BKFB6K94
    TOP 10 HOLDINGS = 'NoneType' object has no attribute 'find' 10
11 == https://www.justetf.com/en/etf-profile.html?isin=IE00BD

In [507]:
url = 'https://www.justetf.com/en/etf-profile.html?isin=LU0671493277#overview'
response = get(url)
if response.status_code != 200:
    print('got bad response !=200')
soup = BeautifulSoup(response.text, 'html.parser')

In [427]:
<table class="table mb-0"> <tbody> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?index=S%2526P%2B500%25C2%25AE%2B%2528EUR%2BHedged%2529&amp;search=ETFS" title="Index: S&amp;P 500® (EUR Hedged)"> <i class="fa fa-search"></i> Index: S&amp;P 500® (EUR Hedged) </button> </td> </tr> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?search=ETFS&amp;assetClass=class-equity&amp;country=US§or=none&amp;equityStrategy=none&amp;theme=none" title="Category: Equity, United States"> <i class="fa fa-search"></i> Category: Equity, United States </button> </td> </tr> </tbody> </table>

<table class="table mb-0"> <tbody> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?index=S%2526P%2B500%25C2%25AE%2B%2528EUR%2BHedged%2529&amp;search=ETFS" title="Index: S&amp;P 500® (EUR Hedged)"> <i class="fa fa-search"></i> Index: S&amp;P 500® (EUR Hedged) </button> </td> </tr> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?search=ETFS&amp;assetClass=class-equity&amp;country=US§or=none&amp;equityStrategy=none&amp;theme=none" title="Category: Equity, United States"> <i class="fa fa-search"></i> Category: Equity, United States </button> </td> </tr> </tbody> </table>

In [426]:
soup.find_all('table', class_='table mb-0')[0]

<table class="table mb-0"> <tbody> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?index=S%2526P%2B500%25C2%25AE%2B%2528EUR%2BHedged%2529&amp;search=ETFS" title="Index: S&amp;P 500® (EUR Hedged)"> <i class="fa fa-search"></i> Index: S&amp;P 500® (EUR Hedged) </button> </td> </tr> <tr> <td> <button class="text-left light-link d-block text-decoration-none p-0" data-js-href="/en/search.html?search=ETFS&amp;assetClass=class-equity&amp;country=US§or=none&amp;equityStrategy=none&amp;theme=none" title="Category: Equity, United States"> <i class="fa fa-search"></i> Category: Equity, United States </button> </td> </tr> </tbody> </table>

In [ ]:
# https://ibkrcampus.com/ibkr-quant-news/stock-market-data-obtaining-data-visualization-analysis-in-python/#:~:text=One%20of%20the%20first%20sources,csv%20file%20by%20using%20pandas.


import yfinance as yf
# Set the start and end date
start_date = '1990-01-01'
end_date = '2021-07-12'
# Set the ticker
ticker = 'AMZN'
# Get the data
data = yf.download(ticker, start_date, end_date)
# Print 5 rows
data.tail()

In [36]:
# # Clear bot check
# def bot_check():
#     while True:
#         sleep(2)
#         user_input = input()
#         if user_input:
#             break

# restart = False

# while True:
#     if restart:
#         restart = False
#         continue
#     try:
#         for result in search_results:
#             try:
#                 if result[0]['query'] in remaining_queries:
#                     remaining_queries.remove(result[0]['query'])
#             except (TypeError, IndexError):
#                 continue
#     except NameError:
#         pass

#     saved_queries += search_results
#     print(len(search_results), len(remaining_queries), len(eur['query'].to_list()))

#     driver = webdriver.Chrome()
#     driver.get('https://www.google.com/search?q=test')

#     # Clear cookies
#     WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".QS5gu.sy4vM")))
#     sleep(random.uniform(0, .1))
#     button = driver.find_element(By.CSS_SELECTOR, ".QS5gu.sy4vM")
#     driver.execute_script("arguments[0].scrollIntoView();", button)
#     sleep(random.uniform(0, .5))
#     button.click()

#     search_results = []
#     for query in remaining_queries:
#         result_data = []

#         driver.get(string_to_url(query))
#         try:
#             # WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".tfyYAe.HqgSZe")))
#             WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".Ap1Qsc")))
#         except TimeoutException:
#             # bot_check()
#             driver.quit()
#             restart = True
#             break
#         if restart:
#             break
#             # continue
#         sleep(random.uniform(0, .5))
#         for i in range(random.randint(0, 2)):
#             actions = ActionChains(driver)
#             sleep(random.uniform(0, .5))
#             actions.send_keys(Keys.PAGE_DOWN).perform()

#         html = driver.page_source
#         soup = BeautifulSoup(html, 'html.parser')
#         results = soup.find_all('div', class_='yuRUbf')

#         for result in results:
#             link = result.find('a')['href']
#             if 'etf-profile' in link:
#                 etf_profile = {
#                     'link': link,
#                     'heading': result.find('h3').text,
#                     'query': query
#                 }
#                 result_data.append(etf_profile)
#             else:
#                 etf_profile = {
#                     'link': None,
#                     'heading': None, 
#                     'query': query
#                 }
#                 result_data.append(etf_profile)

#         search_results.append(result_data)
#         sleep(random.uniform(1, 2))

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=125.0.6422.142)
Stacktrace:
	GetHandleVerifier [0x00007FF6ED001F52+60322]
	(No symbol) [0x00007FF6ECF7CEC9]
	(No symbol) [0x00007FF6ECE37EBA]
	(No symbol) [0x00007FF6ECE0D5A5]
	(No symbol) [0x00007FF6ECEB36B7]
	(No symbol) [0x00007FF6ECECB331]
	(No symbol) [0x00007FF6ECEABFC3]
	(No symbol) [0x00007FF6ECE79617]
	(No symbol) [0x00007FF6ECE7A211]
	GetHandleVerifier [0x00007FF6ED3194AD+3301629]
	GetHandleVerifier [0x00007FF6ED3636D3+3605283]
	GetHandleVerifier [0x00007FF6ED359450+3563680]
	GetHandleVerifier [0x00007FF6ED0B4326+790390]
	(No symbol) [0x00007FF6ECF8750F]
	(No symbol) [0x00007FF6ECF83404]
	(No symbol) [0x00007FF6ECF83592]
	(No symbol) [0x00007FF6ECF72F9F]
	BaseThreadInitThunk [0x00007FF90D45257D+29]
	RtlUserThreadStart [0x00007FF90F08AA48+40]
